# Creating Ground Truth

## Housekeeping (no interaction required)

In [1]:
import os
from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm

tqdm.pandas()

In [2]:
IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ
DATA_DIR = Path('/content/drive/MyDrive/ZBSummerSchool26/data') if IN_COLAB else Path('../data')

In [3]:
def confirm(question: str = "Do you want to execute this cell?"):
    while True:
        response = input(f"{question} (y/n): ").lower()
        if response in ["y", "yes"]:
            return True
        elif response in ["n", "no"]:
            return False
        else:
            print("Please enter 'y' or 'n'.")

## Setup (Interaction required)

In [4]:
### ⬇️⬇️⬇️ 💽 Adjust here if you want to continue with your own query
CORPUS_NAME = "armenpflege"
LOAD_OWN_DATA = True
### ⬆️⬆️⬆️

In [5]:
if IN_COLAB and LOAD_OWN_DATA: # and confirm("Do you want to mount your Google Drive?"):
    from google.colab import drive, files
    drive.mount('/content/drive')
    os.makedirs(DATA_DIR, exist_ok=True)

### Load the data

#### <img src="https://cdn.simpleicons.org/googledrive" alt="💾" width=16> Load your own data from Google Drive

#### <img src="https://cdn.svglogos.dev/logos/google-drive.svg" alt="💾" width=16> Load your own data from Google Drive

In [23]:
if LOAD_OWN_DATA:
    RAWDATA_PATH = DATA_DIR / f"{CORPUS_NAME}.filtered.parquet" # Use data from filtering module
    raw_df = pd.read_parquet(RAWDATA_PATH)

    SENTENCES_PATH = DATA_DIR / f"{CORPUS_NAME}.filtered.sentences.parquet" # Use data from filtering module
    sentences_df = pd.read_parquet(SENTENCES_PATH)

    raw_df = raw_df.join(sentences_df, on="id")

#### <img src="https://cdn.simpleicons.org/github" alt="🏫" width=16> Load from Github

#### <img src="https://www.zb.uzh.ch/themes/zb/assets/images/favicon-192.png" alt="💾" width=16> Load from example

In [22]:
if not LOAD_OWN_DATA:
    # RAWDATA_ORIGIN_URL = "https://github.com/pleyad/Summer-School-2026/raw/refs/heads/main/data/armensteuer_and_similars.filtered.csv" 
    # raw_df = pd.read_csv(RAWDATA_ORIGIN_URL, index_col="id")

    RAWDATA_ORIGIN_URL = "https://github.com/pleyad/Summer-School-2026/raw/refs/heads/main/data/armensteuer_and_similars.filtered.parquet"
    print(f"Loading raw data ...", end="\r")
    raw_df = pd.read_parquet(RAWDATA_ORIGIN_URL)

    SENTENCES_URL = "https://github.com/pleyad/Summer-School-2026/raw/refs/heads/main/data/armensteuer_and_similars.filtered.sentences.parquet"
    print(f"Loading sentences data ...", end="\r")
    sentences_df = pd.read_parquet(SENTENCES_URL)

    raw_df = raw_df.join(sentences_df)

### Create Pseudo-Paragraphs around Search Terms for Exploring and Coding

For this example, we want to label the articles where the search terms are occuring.
To give us more context during labeling, we extract 2 sentences before and 1 sentence after a sentence with a search term.
We call these 4-sentence chunks *pseudo-paragraphs*.

When the LLM is doing the labeling task, we will also only show it these pseudo paragraphs as a *feature*, for three reasons:
1. It is cheaper than sending whole pages.
2. It is more precise. Impresso mainly returned physical *pages*, and not thematic *articles*. A lot of the text has little to do with our search terms.
3. It is simpler. For full-fledged research, you might consider giving the LLM additional context, such as more text or the year of publication. ⚠️ Consider however that this might introduce biases!

This procedure simplifies a lot and has some rough edges:
- Pseudo-paragraphs don't necessarily give enough context to determine the nature of an occurence. We can look at the full text, but the LLM will only see the pseudo-paragraphs.

In [42]:
### ⬇️⬇️⬇️ Adjust here
SEARCH_TERMS = ["armenpflege"] #["armensteuer", "armensteuern", "armenausgaben", "armengut", "armengüter"]
### ⬆️⬆️⬆️

In [43]:
N_BEFORE = 3 # Number of sentences before the sentence containing the search term to include in the pseudo-paragraph
N_AFTER = 3 # Number of sentences after the sentence containing the search term to include in the pseudo-paragraph

from itertools import groupby
from operator import itemgetter

# def create_pseudo_paragraphs(sentences: list[str], search_terms: list[str]) -> list[str]:
#     search_terms = [term.lower() for term in search_terms]
#     pseudo_paragraphs = []
#     for i, sentence in enumerate(sentences):
#         sentence = sentence.lower()
#         if any(term in sentence for term in search_terms):
#             start = max(0, i - N_BEFORE)
#             end = min(len(sentences), i + N_AFTER + 1)
#             pseudo_paragraphs.append(" ".join(sentences[start:end]))
#     return  pseudo_paragraphs

# raw_df["pseudo_paragraph"] = raw_df["_sentences"].progress_apply(lambda s: create_pseudo_paragraphs(s, SEARCH_TERMS))

def create_pseudo_paragraphs(sentences: list[str], search_terms: list[str]) -> str:
    search_terms = [term.lower() for term in search_terms]
    # pseudo_paragraphs = []

    # for i, sentence in enumerate(sentences):
    #     sentence = sentence.lower()
    #     if any(term in sentence for term in search_terms):
    #         start = max(0, i - N_BEFORE)
    #         end = min(len(sentences), i + N_AFTER + 1)
    #         pseudo_paragraphs.append(" ".join(sentences[start:end]))

    hit_indices = [i for i, sentence in enumerate(sentences) if any(term in sentence.lower() for term in search_terms)]
    pseudo_paragraph_indices = []
    for i in hit_indices:
        start = max(0, i - N_BEFORE)
        end = min(len(sentences), i + N_AFTER + 1)
        pseudo_paragraph_indices.extend(range(start, end))
    pseudo_paragraph_indices = sorted(set(pseudo_paragraph_indices))
    
    # Join all consecutive sentences with " "
    pseudo_paragraphs = []
    for _, group in groupby(enumerate(pseudo_paragraph_indices), lambda x: x[0] - x[1]):
        indices = list(map(itemgetter(1), group))
        pseudo_paragraphs.append(" ".join(sentences[i] for i in indices))
    
    return  "   [...]   ".join(pseudo_paragraphs)

raw_df["pseudo_paragraph"] = raw_df["_sentences"].progress_apply(lambda s: create_pseudo_paragraphs(s, SEARCH_TERMS))

  0%|          | 0/2633 [00:00<?, ?it/s]

In [44]:
# p_df = raw_df.explode("pseudo_paragraph")
# p_df = p_df.dropna(subset=["pseudo_paragraph"])
# p_df = p_df[["year", "media_title", "pseudo_paragraph"]]

# # Add a running id suffix to the index ("id") to make it unique after exploding
# p_df["id"] = p_df.index.astype(str) + "_" + p_df.groupby(p_df.index).cumcount().astype(str)
# p_df = p_df.set_index("id")

p_df = raw_df.copy()
#p_df = p_df.dropna(subset=["pseudo_paragraph"])
p_df = p_df[["year", "media_title", "pseudo_paragraph"]]

p_df



,year,media_title,pseudo_paragraph
id,,,
FZG-1932-09-08-a-i0024,1932,Freiburger Nachrichten,als Beitrage an die Kosten für die Unterbringu...
DTT-1969-02-04-a-i0138,1969,Die Tat,"Madame Angele Grammont, eine alte Dame, die be..."
FedGazDe-1891-10-28-a-i0001,1891,Bundesblatt,a. Verwendung des Alkoholzehntels vom Jahre 18...
luxwort-1890-04-09-a-i0027,1890,Luxemburger Wort,Frankreich. Der französische M i n i st e r r ...
NZZ-1898-10-20-a-i0002,1898,Neue Zürcher Zeitung,S. K. -H. 200 Fr. geschenkt worden. Bei den Wa...
...,...,...,...
NZZ-1906-10-16-a-i0001,1906,Neue Zürcher Zeitung,"de, W. llp » fi »«»«» n ».»«.«.«>; « 4.-48. Me..."
NZZ-1899-04-26-b-i0005,1899,Neue Zürcher Zeitung,"Chrysostomus, Johannes der (z. V. in der elfte..."
NZZ-1894-01-22-b-i0002,1894,Neue Zürcher Zeitung,"ungsbudgets."" Von der bernische » Armendirekti..."


In [45]:
p_df = p_df.sample(frac=1, random_state=42)
p_df

,year,media_title,pseudo_paragraph
id,,,
NZZ-1914-07-13-b-i0001,1914,Neue Zürcher Zeitung,In d: r Abstimmung lehnt der Rat die Subventio...
NZZ-1881-07-13-b-i0005,1881,Neue Zürcher Zeitung,Vor zehn Iahren be irgend eine andere protegir...
DTT-1973-12-25-a-i0146,1973,Die Tat,Der Verein will im kommenden Wahlkampf einen e...
NZZ-1883-10-14-a-i0005,1883,Neue Zürcher Zeitung,lichen Zweckc der Gemeinde. Dabei käme ein The...
NZZ-1931-08-19-a-i0001,1931,Neue Zürcher Zeitung,Eine davon betrieb Jahrzehnte hiuduich das Hot...
...,...,...,...
FZG-1958-09-06-a-i0062,1958,Freiburger Nachrichten,Die « N. Z. Z. » berichtet über einen eigenart...
NZZ-1884-06-10-b-i0001,1884,Neue Zürcher Zeitung,"w, «, » 5 »«»»«». «»«««« ^.»«,» 1. »«» O «, » ..."
VHT-1946-08-09-a-i0009,1946,VHTL-Zeitung,Und das hat ungefähr so getönt: * Habt ihr geh...


In [46]:
p_df = p_df.dropna(subset=["pseudo_paragraph"])
p_df = p_df[p_df["pseudo_paragraph"] != ""]

In [47]:
def parse_ids_to_urls(id: str) -> str:
    id_parts = id.split("-")
    article_id = id_parts[-1]
    issue_id = "-".join(id_parts[:-1])
    return f"https://impresso-project.ch/app/issue/{issue_id}/view?p=5&articleId={article_id}"

p_df["url"] = p_df.index.to_series().apply(parse_ids_to_urls)

# Make the url column the first column
cols = p_df.columns.tolist()
cols = cols[-1:] + cols[:-1]
p_df = p_df[cols]

Save the pseudo-paragraphs and download it.

In [ ]:
if LOAD_OWN_DATA: 
    p_df.to_csv(DATA_DIR / f"{CORPUS_NAME}.filtered.pp.csv")
    if IN_COLAB:
        files.download(str(DATA_DIR / f"{CORPUS_NAME}.filtered.pp.csv"))
else:
    print("You can download the pseudo-paragraphs data from the following URL:")
    PP_DATA_URL = "https://github.com/pleyad/Summer-School-2026/blob/main/data/armensteuer_and_similars.filtered.pp.csv"

#### <img src="https://www.zb.uzh.ch/themes/zb/assets/images/favicon-192.png" alt="💾" width=16> <img src="https://cdn.simpleicons.org/github" alt="🏫" width=16> Load from Github

If you're not working with your own data, you may download the pseudo-paragraph-CSV for the *Armensteuer* test case directly on GitHub:

[https://github.com/pleyad/Summer-School-2026/blob/main/data/armensteuer_and_similars.filtered.pseudo_paragraphs.csv](https://github.com/pleyad/Summer-School-2026/blob/main/data/armensteuer_and_similars.filtered.pseudo_paragraphs.csv)

![img](../assets/github_raw_download.png)



## After Annotating

When working with your own data, save your labelled articles on your Google Drive in the Summerschool folder.
The file should have this name:

`{CORPUS_NAME}.filtered.pp.label.csv`

To export the File as `csv`, you need explicitly change the filetype during saving.
⚠️ You need to choose **CSV UTF-8**, and not any other csv-dialect!

![Save table as csv in Excel](../assets/save_excel_as_csv.png)

If you are annotating the data in Excel, you can use the following VBA code to set your search terms in bold for easier reading.

```vb
Sub BoldOccurrencesInSelection()
    Dim searchText As String
    Dim cell As Range
    Dim startPos As Long
    Dim textLen As Long
    Dim count As Long

    ' Check something is selected
    If Selection Is Nothing Then
        MsgBox "No cells selected."
        Exit Sub
    End If

    ' Prompt user for the search string
    searchText = InputBox("Enter the text to make bold:", "Bold Text in Selection")

    If searchText = "" Then
        MsgBox "No text entered. Macro cancelled."
        Exit Sub
    End If

    textLen = Len(searchText)
    count = 0

    ' Loop through selected cells only
    For Each cell In Selection
        If cell.HasFormula = False And InStr(1, cell.Value, searchText, vbTextCompare) > 0 Then
            startPos = 1
            Do
                startPos = InStr(startPos, cell.Value, searchText, vbTextCompare)
                If startPos = 0 Then Exit Do
                cell.Characters(startPos, textLen).Font.Bold = True
                count = count + 1
                startPos = startPos + textLen
            Loop
        End If
    Next cell

    MsgBox count & " occurrence(s) of """ & searchText & """ made bold in selection."
End Sub
```